In [1]:
import pandas as pd

In [8]:
import os

len(os.listdir('ISIC2018_Task1-2_Training_Input')) == len(os.listdir('ISIC2018_Task1_Training_GroundTruth'))

True

In [21]:
# Check image sizes
from collections import Counter
from PIL import Image
import numpy as np
import os

img_dir = "ISIC2018_Task1_Training_GroundTruth"

sizes = Counter()
mask = []

with Image.open(f'{img_dir}/ISIC_0000126_segmentation.png') as img:
    sizes[img.size] += 1
    mask.append(np.unique(np.array(img)))

print(sizes)
print(mask)

Counter({(2048, 1536): 1})
[array([  0, 255], dtype=uint8)]


In [1]:
from collections import Counter
import os
import numpy as np
from PIL import Image

img_dir  = "ISIC2018_Task1-2_Training_Input"
mask_dir = "ISIC2018_Task1_Training_GroundTruth"

img_sizes   = Counter()
img_pixels  = []
mask_pixels = []

for file in os.listdir(img_dir):
    if file.endswith(".jpg"):
        # Image sizes and pixel ranges
        with Image.open(os.path.join(img_dir, file)) as img:
            img_sizes[img.size] += 1
            arr = np.array(img)
            img_pixels.append((arr.min(), arr.max()))

        # Mask pixel values
        mask_file = file.replace('.jpg', '_segmentation.png')
        with Image.open(os.path.join(mask_dir, mask_file)) as mask:
            mask_pixels.append(np.unique(np.array(mask)))

print("Image sizes:", img_sizes)
print(f"Image pixel range — min: {min(p[0] for p in img_pixels)}, max: {max(p[1] for p in img_pixels)}")
print("Unique mask values:", np.unique(np.concatenate(mask_pixels)))

Image sizes: Counter({(4288, 2848): 558, (1024, 768): 556, (3008, 2000): 379, (3872, 2592): 125, (3024, 2016): 100, (767, 576): 63, (2048, 1536): 56, (3072, 2304): 53, (1504, 1129): 50, (919, 802): 49, (2592, 1936): 34, (6688, 4439): 32, (6668, 4439): 30, (1022, 767): 20, (6688, 4459): 19, (6708, 4439): 19, (6648, 4439): 19, (824, 719): 16, (2304, 1536): 16, (6668, 4459): 11, (1502, 1051): 10, (6648, 4419): 10, (6642, 4420): 9, (6642, 4422): 9, (962, 674): 8, (6648, 4459): 8, (6641, 4420): 8, (6708, 4459): 8, (722, 542): 7, (6628, 4419): 7, (6688, 4419): 7, (6621, 4420): 7, (2592, 1944): 6, (6621, 4401): 6, (576, 768): 5, (6628, 4399): 5, (6641, 4440): 5, (6668, 4419): 5, (6622, 4441): 5, (6641, 4441): 5, (2816, 2112): 4, (1028, 753): 4, (6642, 4401): 4, (6642, 4421): 4, (6661, 4441): 4, (6642, 4441): 4, (6601, 4380): 4, (6601, 4382): 4, (6601, 4401): 4, (6681, 4422): 4, (6662, 4422): 4, (962, 722): 3, (2272, 1704): 3, (1019, 717): 3, (6602, 4361): 3, (6682, 4401): 3, (6628, 4439): 3, 

### Build the dataframes with the images and labels

In [32]:
import os
import pandas as pd

df = pd.DataFrame()

training_folder = 'ISIC2018_Task1-2_Training_Input'
groundtruth = 'ISIC2018_Task1_Training_GroundTruth'

img_ids = [f.replace('.jpg', '') for f in os.listdir(training_folder) if f.endswith('.jpg')]

df = pd.DataFrame({
    'feature': [os.path.join(training_folder, f'{i}.jpg') for i in img_ids],
    'target':  [os.path.join(groundtruth, f'{i}_segmentation.png') for i in img_ids]
})
pd.set_option('display.max_colwidth', None)
df.shape
df


,feature,target
0,ISIC2018_Task1-2_Training_Input\ISIC_0000000.jpg,ISIC2018_Task1_Training_GroundTruth\ISIC_0000000_segmentation.png
1,ISIC2018_Task1-2_Training_Input\ISIC_0000001.jpg,ISIC2018_Task1_Training_GroundTruth\ISIC_0000001_segmentation.png
2,ISIC2018_Task1-2_Training_Input\ISIC_0000003.jpg,ISIC2018_Task1_Training_GroundTruth\ISIC_0000003_segmentation.png
3,ISIC2018_Task1-2_Training_Input\ISIC_0000004.jpg,ISIC2018_Task1_Training_GroundTruth\ISIC_0000004_segmentation.png
4,ISIC2018_Task1-2_Training_Input\ISIC_0000006.jpg,ISIC2018_Task1_Training_GroundTruth\ISIC_0000006_segmentation.png
...,...,...
2589,ISIC2018_Task1-2_Training_Input\ISIC_0016068.jpg,ISIC2018_Task1_Training_GroundTruth\ISIC_0016068_segmentation.png
2590,ISIC2018_Task1-2_Training_Input\ISIC_0016069.jpg,ISIC2018_Task1_Training_GroundTruth\ISIC_0016069_segmentation.png
2591,ISIC2018_Task1-2_Training_Input\ISIC_0016070.jpg,ISIC2018_Task1_Training_GroundTruth\ISIC_0016070_segmentation.png
2592,ISIC2018_Task1-2_Training_Input\ISIC_0016071.jpg,ISIC2018_Task1_Training_GroundTruth\ISIC_0016071_segmentation.png


### Train, Val, Test Splitting

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(df['feature'], df['target'], train_size = 0.8, shuffle=True, random_state=42) 

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, train_size = 0.5, shuffle=True, random_state=42) 

print(X_train.shape, X_val.shape, X_test.shape)

(2075,) (259,) (260,)


In [1]:
import wandb

wandb.login(key='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')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\nasos\_netrc
wandb: Currently logged in as: nasosk16 (nasosk16-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [1]:
import torch
torch.load('best_model.pth')

OrderedDict([('enc.patchembed.split.weight',
              tensor([[[[ 0.0730, -0.0633, -0.0182, -0.2108],
                        [ 0.2600, -0.1252,  0.0045, -0.0024],
                        [-0.0532,  0.1128,  0.1359,  0.1426],
                        [-0.1135, -0.2182,  0.0495, -0.1732]]],
              
              
                      [[[-0.0959, -0.0020,  0.1618,  0.1576],
                        [-0.1460, -0.0343, -0.1766, -0.2603],
                        [ 0.1088,  0.1290, -0.0924,  0.1858],
                        [-0.0967,  0.1793,  0.2602, -0.0635]]],
              
              
                      [[[-0.2427, -0.0648, -0.1486,  0.0974],
                        [-0.0349, -0.0185,  0.0507, -0.2115],
                        [ 0.0418,  0.1909, -0.0253,  0.2369],
                        [ 0.1976, -0.1794,  0.0285, -0.2288]]],
              
              
                      ...,
              
              
                      [[[ 0.1231, -0.1883,  0.0192, -0.255

In [5]:
import os
import numpy as np
from PIL import Image

img_dir = 'frames'
means, stds = [], []
for file in os.listdir(img_dir):
    if file.endswith('.png'):
        img = np.array(Image.open(os.path.join(img_dir, file)).convert('L')) / 255.0
        means.append(img.mean())
        stds.append(img.std())

mean = np.mean(means)
std  = np.mean(stds)
print(f"Mean: {mean:.3f}, Std: {std:.3f}")

Mean: 0.283, Std: 0.217


In [1]:
from train import train_dataloader
images, masks = next(iter(train_dataloader))
print(masks.unique())  # must be tensor([0., 1.])

ImportError: cannot import name 'train_dataloader' from 'train' (c:\Users\nasos\Desktop\CITY MSC\SEMESTER 2\DL FOR IMAGE ANALYSIS\DLImageAnalysis\train.py)